### Import modules

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


2025-12-23 08:23:23.037227: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Import Data

In [2]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
data['pattern'].fillna('None', inplace=True)

synthetic_data       = data[data['file'].str.contains('pattern',na=False)]
verified_communities = data[~data['file'].str.contains('pattern',na=False)]

/tmp/ipykernel_12543/859644980.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['pattern'].fillna('None', inplace=True)


### Custom Train Test Split

In [3]:
def get_folded_splits(fold_count=4,random_state=42,use_only_verified=True,min_samples_per_class=5):
    folded_data = []

    _sd = synthetic_data.copy()
    _vd = verified_communities.copy()

    _vd_c = _vd['pattern'].value_counts()
    _vp_i = _vd_c[_vd_c>=min_samples_per_class].index.tolist()
    _vd   = _vd[_vd['pattern'].isin(_vp_i)]

    _vd_kfold = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=random_state)
    _vd_folds = _vd_kfold.split(_vd, _vd['pattern'])

    for train_indices, test_indices in _vd_folds:
        vd_train = _vd.iloc[train_indices]
        vd_test = _vd.iloc[test_indices]
        if use_only_verified:
            train_data = vd_train
            test_data  = vd_test
        else:
            train_data = pd.concat([vd_train,_sd])
            test_data  = vd_test 

        folded_data.append((train_data, test_data))
    return folded_data


# def get_folded_splits(fold_count=4,random_state=42,verified_train_set=True):
#     folded_data = []

#     _sd = synthetic_data.copy()
#     _vd = verified_communities.copy()

#     _vd_grouped = _vd.groupby('pattern')

#     _sd_counts = _sd['pattern'].value_counts()
#     _vd_counts = _vd['pattern'].value_counts()

#     verified_data = pd.DataFrame()

#     for label, vd_group in _vd_grouped:
#         total_count = _sd_counts.get(label,0) + _vd_counts.get(label,0)
#         max_train_count = int(total_count/fold_count*(fold_count-1 if verified_train_set else 1))

#         vd_count = len(vd_group)
#         selected_indexes = np.random.choice(vd_group.index,size=min(vd_count, max_train_count),replace=False)
#         verified_data = pd.concat([verified_data, vd_group.loc[selected_indexes]])
#         _vd = _vd.drop(index=selected_indexes)

#     kfold = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=random_state)
#     folds = kfold.split(_sd, _sd['pattern'])

#     for train_indices, test_indices in folds:
#         sd_train_data = _sd.iloc[train_indices]
#         sd_test_data  = _sd.iloc[test_indices]
#         if verified_train_set:
#             train_data = pd.concat([sd_train_data, verified_data])
#             test_data  = pd.concat([sd_test_data, _vd])
#         else:
#             train_data = pd.concat([sd_train_data,_vd ])
#             test_data  = pd.concat([sd_test_data,verified_data])
            
#         folded_data.append((train_data, test_data))
    
#     return folded_data


### Preprocess

In [4]:
TARGET_COLUMN = 'pattern'

In [5]:
def prepare_data(train_data,test_data):
    numeric_features = train_data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected - please ensure embeddings/features are present.")

    X_train = train_data[numeric_features].fillna(0.0).values
    X_temp = test_data[numeric_features].fillna(0.0).values
    y_train = train_data[TARGET_COLUMN].astype(str).values
    y_temp = test_data[TARGET_COLUMN].astype(str).values

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)
    y_temp_enc = label_encoder.transform(y_temp)
    num_classes = len(label_encoder.classes_)


    return X_train, X_temp, y_train_enc, y_temp_enc, label_encoder, num_classes

In [6]:
def preprocess_data_nn(train_data,test_data):
    X_train, X_temp, y_train_enc, y_temp_enc, label_encoder, num_classes = prepare_data(train_data,test_data)

    X_val, X_test, y_val_enc, y_test_enc = train_test_split(
        X_temp,
        y_temp_enc,
        test_size=0.5,
        random_state=42,
        stratify=y_temp_enc,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)
    return X_train, X_val, X_test, y_train, y_val, y_test,y_test_enc, label_encoder, num_classes, scaler

### NN Model

In [7]:
def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            Dense(num_classes, activation="softmax"),
        ]
    )

In [ ]:
kfolded_data = get_folded_splits(fold_count=3,use_only_verified=False,min_samples_per_class=8)
fold_results = []
predictions = []
actuals = []
for fold_index, (train_data, test_data) in enumerate(kfolded_data):
    print(f"Processing Fold {fold_index + 1}")

    X_train, X_val, X_test, y_train, y_val, y_test,y_test_enc, label_encoder, num_classes, scaler = preprocess_data_nn(train_data,test_data)

    model = build_classifier(X_train.shape[1], num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=0,
    )

    test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

    y_pred = model.predict(X_test)
    y_pred_labels = y_pred.argmax(axis=1)

    report = classification_report(
        y_test_enc,
        y_pred_labels,
        target_names=label_encoder.inverse_transform(np.unique(np.concatenate((y_test_enc, y_pred_labels)))),
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    fold_results.append({
        "summary": summary,
        "class_breakdown": class_breakdown,
        "history": history.history,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_top3": test_top3,
    })
    actuals.extend(y_test_enc.tolist())
    predictions.extend(y_pred_labels.tolist())



overall_report = classification_report(
    actuals,
    predictions,
    target_names=label_encoder.inverse_transform(np.unique(np.concatenate((actuals, predictions)))),
    output_dict=True,
    zero_division=0,
)

print("Accuracy:", overall_report["accuracy"])
print("F1 Score:", overall_report["weighted avg"]["f1-score"])

display(pd.DataFrame(overall_report).T)

Processing Fold 1
Test loss: 2.2706 | Test accuracy: 0.3469 | Test top-3 accuracy: 0.6531
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Processing Fold 2
Test loss: 3.4987 | Test accuracy: 0.3469 | Test top-3 accuracy: 0.6939
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
Processing Fold 3
Test loss: 2.5021 | Test accuracy: 0.4375 | Test top-3 accuracy: 0.7500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
Accuracy: 0.3767123287671233
F1 Score: 0.3814014245521095


,precision,recall,f1-score,support
Advanced LLM Prompting,0.000000,0.000000,0.000000,3.000000
Classical Models,0.523810,0.458333,0.488889,24.000000
Enhanced User Intent Comprehension with LLMs,0.333333,0.250000,0.285714,4.000000
Explainable AI (XAI) Techniques,0.000000,0.000000,0.000000,0.000000
Integrating External Knlowladge with LLM,0.000000,0.000000,0.000000,0.000000
LLM Code Execution for Precision,0.000000,0.000000,0.000000,0.000000
LLM Context Management,0.000000,0.000000,0.000000,0.000000
LLM Results Evaluation,0.000000,0.000000,0.000000,6.000000
LLM based Multimodal Generative Prompting,0.538462,0.538462,0.538462,13.000000
"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",0.000000,0.000000,0.000000,3.000000


### Logistic Regression

In [80]:
kfolded_data = get_folded_splits(fold_count=3,use_only_verified=True,min_samples_per_class=8)

fold_results = []
predictions = []
actuals = []
for fold_index, (train_data, test_data) in enumerate(kfolded_data):
    print(f"Processing Fold {fold_index + 1}")

    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X_train = train_data[numeric_features].fillna(0.0).values
    X_test = test_data[numeric_features].fillna(0.0).values
    y_train = train_data[TARGET_COLUMN].astype(str).values
    y_test = test_data[TARGET_COLUMN].astype(str).values

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)
    logreg.fit(X_train, y_train_encoded)

    y_pred = logreg.predict(X_test)
    report = classification_report(
        y_test_encoded,
        y_pred,
        target_names=label_encoder.inverse_transform(np.unique(np.concatenate((y_test_encoded, y_pred)))),
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    fold_results.append({
        "summary": summary,
        "class_breakdown": class_breakdown,
    })
    actuals.extend(y_test_encoded.tolist())
    predictions.extend(y_pred.tolist())

overall_report = classification_report(
    actuals,
    predictions,
    target_names=label_encoder.inverse_transform(np.unique(np.concatenate((actuals, predictions)))),
    output_dict=True,
    zero_division=0,
)

print("Accuracy:", overall_report["accuracy"])
print("F1 Score:", overall_report["weighted avg"]["f1-score"])

pd.DataFrame(overall_report).T

Processing Fold 1


Processing Fold 2
Processing Fold 3
Accuracy: 0.4241379310344828
F1 Score: 0.40779550395960007


,precision,recall,f1-score,support
Advanced LLM Prompting,0.000000,0.000000,0.000000,8.000000
Classical Models,0.559322,0.702128,0.622642,47.000000
Enhanced User Intent Comprehension with LLMs,0.600000,0.375000,0.461538,8.000000
LLM Results Evaluation,0.142857,0.076923,0.100000,13.000000
LLM based Multimodal Generative Prompting,0.461538,0.461538,0.461538,26.000000
"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",0.250000,0.125000,0.166667,8.000000
Model Abstraction Pattern,0.266667,0.250000,0.258065,16.000000
Modular LLM Agent Architectures,0.333333,0.250000,0.285714,20.000000
None,0.450000,0.562500,0.500000,64.000000
Preprocessing Text and Numerical Data,0.400000,0.303030,0.344828,33.000000


In [81]:
kfolded_data = get_folded_splits(fold_count=3,use_only_verified=False,min_samples_per_class=8)

fold_results = []
predictions = []
actuals = []
for fold_index, (train_data, test_data) in enumerate(kfolded_data):
    print(f"Processing Fold {fold_index + 1}")

    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X_train = train_data[numeric_features].fillna(0.0).values
    X_test = test_data[numeric_features].fillna(0.0).values
    y_train = train_data[TARGET_COLUMN].astype(str).values
    y_test = test_data[TARGET_COLUMN].astype(str).values

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)
    logreg.fit(X_train, y_train_encoded)

    y_pred = logreg.predict(X_test)
    report = classification_report(
        y_test_encoded,
        y_pred,
        target_names=label_encoder.inverse_transform(np.unique(np.concatenate((y_test_encoded, y_pred)))),
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T
    summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
    class_breakdown = (
        report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
    )
    fold_results.append({
        "summary": summary,
        "class_breakdown": class_breakdown,
    })
    actuals.extend(y_test_encoded.tolist())
    predictions.extend(y_pred.tolist())

overall_report = classification_report(
    actuals,
    predictions,
    target_names=label_encoder.inverse_transform(np.unique(np.concatenate((actuals, predictions)))),
    output_dict=True,
    zero_division=0,
)

print("Accuracy:", overall_report["accuracy"])
print("F1 Score:", overall_report["weighted avg"]["f1-score"])

pd.DataFrame(overall_report).T

Processing Fold 1
Processing Fold 2
Processing Fold 3
Accuracy: 0.41379310344827586
F1 Score: 0.4218979157980347


,precision,recall,f1-score,support
Advanced LLM Prompting,0.142857,0.125000,0.133333,8.000000
Classical Models,0.574468,0.574468,0.574468,47.000000
Enhanced User Intent Comprehension with LLMs,0.285714,0.250000,0.266667,8.000000
Integrating External Knlowladge with LLM,0.000000,0.000000,0.000000,0.000000
LLM Code Execution for Precision,0.000000,0.000000,0.000000,0.000000
LLM Context Management,0.000000,0.000000,0.000000,0.000000
LLM KV Cache Optimization,0.000000,0.000000,0.000000,0.000000
LLM Results Evaluation,0.181818,0.153846,0.166667,13.000000
LLM based Multimodal Generative Prompting,0.576923,0.576923,0.576923,26.000000
"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",0.181818,0.500000,0.266667,8.000000


### Dataset Analysis

In [60]:
kfolded_data = get_folded_splits(fold_count=3,use_only_verified=True,min_samples_per_class=10)
for i in kfolded_data:
    print(i[0].shape, i[1].shape)
    counts_train = i[0]['pattern'].value_counts()
    counts_test = i[1]['pattern'].value_counts()
    fold_summary = pd.DataFrame({'train_counts': counts_train, 'test_counts': counts_test}).fillna(0).astype(int).sort_values(by='test_counts', ascending=False)
    display(fold_summary)

Using patterns: ['None', 'Classical Models', 'Preprocessing Text and Numerical Data', 'Tool Use for LLMs', 'LLM based Multimodal Generative Prompting', 'Retrieval Augmented Generation(RAG) Optimization for LLMs', 'Modular LLM Agent Architectures', 'Model Abstraction Pattern', 'LLM Results Evaluation']
(177, 770) (89, 770)


,train_counts,test_counts
pattern,,
None,43,21
Classical Models,31,16
Preprocessing Text and Numerical Data,22,11
LLM based Multimodal Generative Prompting,17,9
Tool Use for LLMs,18,9
Modular LLM Agent Architectures,13,7
Retrieval Augmented Generation(RAG) Optimization for LLMs,13,7
Model Abstraction Pattern,11,5
LLM Results Evaluation,9,4


(177, 770) (89, 770)


,train_counts,test_counts
pattern,,
None,43,21
Classical Models,31,16
Preprocessing Text and Numerical Data,22,11
LLM based Multimodal Generative Prompting,17,9
Tool Use for LLMs,18,9
Modular LLM Agent Architectures,13,7
Retrieval Augmented Generation(RAG) Optimization for LLMs,13,7
Model Abstraction Pattern,11,5
LLM Results Evaluation,9,4


(178, 770) (88, 770)


,train_counts,test_counts
pattern,,
None,42,22
Classical Models,32,15
Preprocessing Text and Numerical Data,22,11
Tool Use for LLMs,18,9
LLM based Multimodal Generative Prompting,18,8
Model Abstraction Pattern,10,6
Modular LLM Agent Architectures,14,6
Retrieval Augmented Generation(RAG) Optimization for LLMs,14,6
LLM Results Evaluation,8,5


In [61]:
kfolded_data = get_folded_splits(fold_count=3,use_only_verified=True,min_samples_per_class=10)
for i in kfolded_data:
    print(i[0].shape, i[1].shape)
    counts_train = i[0]['pattern'].value_counts()
    counts_test = i[1]['pattern'].value_counts()
    fold_summary = pd.DataFrame({'train_counts': counts_train, 'test_counts': counts_test}).fillna(0).astype(int).sort_values(by='test_counts', ascending=False)
    display(fold_summary)

Using patterns: ['None', 'Classical Models', 'Preprocessing Text and Numerical Data', 'Tool Use for LLMs', 'LLM based Multimodal Generative Prompting', 'Retrieval Augmented Generation(RAG) Optimization for LLMs', 'Modular LLM Agent Architectures', 'Model Abstraction Pattern', 'LLM Results Evaluation']
(177, 770) (89, 770)


,train_counts,test_counts
pattern,,
None,43,21
Classical Models,31,16
Preprocessing Text and Numerical Data,22,11
LLM based Multimodal Generative Prompting,17,9
Tool Use for LLMs,18,9
Modular LLM Agent Architectures,13,7
Retrieval Augmented Generation(RAG) Optimization for LLMs,13,7
Model Abstraction Pattern,11,5
LLM Results Evaluation,9,4


(177, 770) (89, 770)


,train_counts,test_counts
pattern,,
None,43,21
Classical Models,31,16
Preprocessing Text and Numerical Data,22,11
LLM based Multimodal Generative Prompting,17,9
Tool Use for LLMs,18,9
Modular LLM Agent Architectures,13,7
Retrieval Augmented Generation(RAG) Optimization for LLMs,13,7
Model Abstraction Pattern,11,5
LLM Results Evaluation,9,4


(178, 770) (88, 770)


,train_counts,test_counts
pattern,,
None,42,22
Classical Models,32,15
Preprocessing Text and Numerical Data,22,11
Tool Use for LLMs,18,9
LLM based Multimodal Generative Prompting,18,8
Model Abstraction Pattern,10,6
Modular LLM Agent Architectures,14,6
Retrieval Augmented Generation(RAG) Optimization for LLMs,14,6
LLM Results Evaluation,8,5
